# Selective Gaussian Number Splitting


This companion notebook focuses on interpretation rather than pulse construction. It reuses the calibrated displacement-plus-spectroscopy workflow and asks a narrower question: when do the resolved peak heights become faithful proxies for cavity photon-number weights?

The answer in this regime is: when the qubit probe is weak and spectrally selective enough that each dispersive manifold produces its own approximately Gaussian line. Then the measured peak amplitudes track the displaced-state Fock distribution.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from examples.displacement_qubit_spectroscopy import run_displacement_then_qubit_spectroscopy
from tutorials.tutorial_support import coherent_state_fock_probabilities


## Physics / model definition


In [ ]:
result = run_displacement_then_qubit_spectroscopy()
alpha = complex(
    result["displacement"]["target_alpha_real"],
    result["displacement"]["target_alpha_imag"],
)
predicted_lines_mhz = np.asarray(result["predicted_transition_lines_mhz"], dtype=float)
predicted_weights = np.asarray(result["predicted_line_weights"], dtype=float)
normalized_peak_weights = np.asarray(result["normalized_peak_weights"], dtype=float)
ideal_poisson = coherent_state_fock_probabilities(alpha, predicted_weights.size)
ideal_poisson = ideal_poisson / max(float(np.sum(ideal_poisson)), 1.0e-15)


## Pulse / sequence construction


In [ ]:
print("This notebook reuses the calibrated displacement and selective Gaussian probe from the workflow example so that the interpretation stays aligned with the published docs image.")
print("Target alpha:", alpha)
print("Peak-weight correlation:", result["peak_weight_correlation"])


## Simulation


In [ ]:
transition_scan_mhz = np.asarray(result["spectroscopy"]["transition_detunings_mhz"], dtype=float)
spectroscopy_response = np.asarray(result["spectroscopy"]["pe_final"], dtype=float)
theory_response = np.asarray(result["spectroscopy"]["theory_response"], dtype=float)
nbar = float(result["displacement"]["nbar_after_displacement"])


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.0))

axes[0].plot(transition_scan_mhz, spectroscopy_response, color="#4C78A8", lw=1.5, label="pulse-level simulation")
axes[0].plot(transition_scan_mhz, theory_response, "--", color="black", lw=1.1, label="weak-drive Gaussian theory")
for line_mhz in predicted_lines_mhz:
    axes[0].axvline(line_mhz, color="#E45756", alpha=0.35, lw=0.9)
axes[0].set_xlabel("Qubit transition detuning relative to frame (MHz)")
axes[0].set_ylabel("Final excited-state probability $P_e$")
axes[0].set_title("Resolved manifolds under a selective Gaussian probe")
axes[0].legend(loc="upper left")

levels = np.arange(predicted_weights.size, dtype=int)
width = 0.25
axes[1].bar(levels - width, predicted_weights, width=width, color="#72B7B2", label="simulated displaced-state weight")
axes[1].bar(levels, normalized_peak_weights, width=width, color="#F58518", label="normalized peak height")
axes[1].bar(levels + width, ideal_poisson, width=width, color="#54A24B", label="ideal coherent-state Poisson weight")
axes[1].set_xlabel("Photon number manifold $n$")
axes[1].set_ylabel("Normalized weight")
axes[1].set_title(rf"Weight recovery for $D(\alpha={np.real(alpha):+.1f}{np.imag(alpha):+.1f}i)$ with $\langle n \rangle={nbar:.2f}$")
axes[1].set_xticks(levels)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()


## Interpretation


The simulated displaced-state weights and the resolved peak heights agree closely because the probe is long, weak, and Gaussian. In that limit the qubit is effectively sampling one dispersive manifold at a time.

The ideal coherent-state Poisson bars are a useful reference, but the physically relevant comparison is the simulation's actual post-displacement cavity state. Small differences between the two come from finite truncation and pulse-level preparation details.


## Variations / exercises


- Shorten the probe duration and watch the peaks broaden into one another.
- Increase the probe amplitude and check when the weak-drive Gaussian theory starts to fail.
- Change the target displacement amplitude and confirm that the recovered weights move with the expected Poisson mean $|\alpha|^2$.
